In [ ]:
import pandas as pd
import numpy as np

In [2]:
movie = pd.read_csv('tmdb_5000_movies.csv')

In [3]:
credits = pd.read_csv('tmdb_5000_credits.csv')

In [ ]:
movie.head()

In [ ]:
movie.shape

In [ ]:
credits.head()

In [ ]:
credits.shape

In [8]:
movies = movie.merge(credits,on='title')

In [ ]:
movies.shape

In [ ]:
movies.head(1)

In [11]:
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [ ]:
movies.head()

In [ ]:
movies.isnull().sum()

In [14]:
movies.dropna(inplace=True)

In [ ]:
movies.duplicated().sum()

In [16]:
import ast

In [17]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name']) 
    return L 

In [ ]:
movies['genres'] = movies['genres'].apply(convert)
movies.head()

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)
movies.head()

In [20]:
def convert3(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
        counter+=1
    return L

In [ ]:
movies['cast'] = movies['cast'].apply(convert3)
movies.head(1)

In [22]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L 

In [23]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies.head()

In [25]:
movies['overview']= movies['overview'].apply(lambda x:x.split())

In [ ]:
movies.head()

In [27]:
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

In [28]:
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

In [ ]:
movies.head()

In [30]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [ ]:
movies.head()

In [32]:
new = movies.drop(columns=['overview','genres','keywords','cast','crew'])

In [33]:
new = movies[['movie_id','title','tags']].copy()

In [ ]:
new['tags'] = new['tags'].apply(lambda x: " ".join(x))
new.head()

In [35]:
new['tags'] = new['tags'].apply(lambda x:x.lower())

In [ ]:
new.head()

In [37]:
import nltk 
from nltk.stem.porter import PorterStemmer
ps= PorterStemmer()

In [38]:
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [39]:
new['tags'] = new['tags'].apply(stem)

In [ ]:
new['tags'][1400]

In [41]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')

In [42]:
vector = cv.fit_transform(new['tags']).toarray()

In [ ]:
vector

In [ ]:
cv.get_feature_names_out()

In [ ]:
vector.shape

In [46]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
cosine_similarity(vector).shape

In [48]:
similarity = cosine_similarity(vector)

In [ ]:
similarity

In [50]:
def recommend(movie):
    movie = movie.lower().strip()
    
    titles = new['title'].str.lower().str.strip()
    
    if movie not in titles.values:
        print("Movie not found")
        return
    
    movie_index = titles[titles == movie].index[0]
    distances = similarity[movie_index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1])[1:6]

    for i in movies_list:
        print(new.iloc[i[0]].title)

In [ ]:
recommend('Batman Begins')

In [55]:
import pickle

pickle.dump(new, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))